In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/anthropic/test.jsonl
/kaggle/input/anthropic/train.jsonl


In [4]:
!pip install jsonlines


In [5]:
!pip install trl



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 2.0 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 46.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 23.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [11]:
!pip install -U transformers


In [7]:
pip install peft


Note: you may need to restart the kernel to use updated packages.


In [12]:
import jsonlines
import pandas as pd

data = []
with jsonlines.open("/kaggle/input/anthropic/train.jsonl") as reader:
    for obj in reader:
        data.append(obj)

df = pd.DataFrame(data)
df.head()


,chosen,rejected
0,"\n\nHuman: If I was arrested for a crime, what...","\n\nHuman: If I was arrested for a crime, what..."
1,\n\nHuman: What tools do I need to work on a c...,\n\nHuman: What tools do I need to work on a c...
2,\n\nHuman: What are some good exercises I can ...,\n\nHuman: What are some good exercises I can ...
3,\n\nHuman: I'm moving to Utah next month and I...,\n\nHuman: I'm moving to Utah next month and I...
4,\n\nHuman: What is an easy to make cake frosti...,\n\nHuman: What is an easy to make cake frosti...


In [13]:
def extract_prompt(text):
    if "Assistant:" in text:
        return text.split("Assistant:")[0].replace("Human:", "").strip()
    return ""

def remove_prompt(text):
    if "Assistant:" in text:
        return text.split("Assistant:")[1].strip()
    return text.strip()

df["prompt"] = df["chosen"].apply(extract_prompt)
df["chosen"] = df["chosen"].apply(remove_prompt)
df["rejected"] = df["rejected"].apply(remove_prompt)


In [14]:
df[["prompt", "chosen", "rejected"]].to_json("dpo_ready_train.jsonl", orient="records", lines=True)


In [15]:
data = []
with jsonlines.open("/kaggle/working/dpo_ready_train.jsonl") as reader:
    for obj in reader:
        data.append(obj)

df = pd.DataFrame(data)
df.head()

,prompt,chosen,rejected
0,"If I was arrested for a crime, what rights do ...","Human: I am not a lawyer, so I don’t have much...","Human: I am not a lawyer, so I don’t have much..."
1,What tools do I need to work on a car?,Generally you’ll need tools like wrenches and ...,Generally you’ll need tools like wrenches and ...
2,What are some good exercises I can do at a des...,You could try some of these out:\n\n\n-Avoid s...,Great question! Here are a few tips. It’s al...
3,I'm moving to Utah next month and I'm curious ...,I think there are a lot of fantastic climbing ...,I think there are a lot of fantastic climbing ...
4,What is an easy to make cake frosting?,"Frosting is a topping for cakes, usually made ...","Frosting is a topping for cakes, usually made ..."


In [16]:
# Kiểm tra xem có NaN hay không
print(df.isnull().sum())

# Kiểm tra số dòng mà chosen == rejected
print((df['chosen'] == df['rejected']).sum())


prompt      0
chosen      0
rejected    0
dtype: int64
35687


In [20]:
# Lọc bỏ các dòng có chosen == rejected
df_cleaned = df[df['chosen'] != df['rejected']].reset_index(drop=True)
df_cleaned.to_json("/kaggle/working/dpo_ready_filtered.jsonl", orient="records", lines=True)


print(f"Số dòng ban đầu: {len(df)}")
print(f"Số dòng sau khi lọc: {len(df_cleaned)}")




Số dòng ban đầu: 52421
Số dòng sau khi lọc: 16734


In [21]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/kaggle/working/dpo_ready_filtered.jsonl", split="train")

dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]
eval_dataset.to_json("/kaggle/working/dpo_eval.jsonl", orient="records", lines=True)

Generating train split: 0 examples [00:00, ? examples/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

1354277

In [22]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

# === Load model nhỏ nhẹ để token hóa ===
model_name = "EleutherAI/pythia-70m"  # hoặc: "tiiuae/falcon-rw-1b"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Đảm bảo tokenizer có pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === Load dữ liệu gốc (file JSONL train) ===
df = pd.read_json("/kaggle/working/dpo_ready_filtered.jsonl", lines=True)
hf_dataset = Dataset.from_pandas(df)

# === Hàm tokenize chuẩn cho DPO ===
def dpo_tokenize(example):
    prompt = example["prompt"]
    chosen = example["chosen"]
    rejected = example["rejected"]

    # Tokenize prompt + chosen
    chosen_enc = tokenizer(
        prompt,
        chosen,
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    # Tokenize prompt + rejected
    rejected_enc = tokenizer(
        prompt,
        rejected,
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    return {
        "prompt_chosen_input_ids": chosen_enc["input_ids"],
        "prompt_chosen_attention_mask": chosen_enc["attention_mask"],
        "prompt_rejected_input_ids": rejected_enc["input_ids"],
        "prompt_rejected_attention_mask": rejected_enc["attention_mask"],
    }

# === Áp dụng tokenize toàn bộ dataset ===
tokenized_dataset = hf_dataset.map(dpo_tokenize, batched=False)

# === Lưu ra file nếu cần (ở Kaggle working) ===
tokenized_dataset.to_json("/kaggle/working/tokenized_dpo.json")


Map:   0%|          | 0/16734 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/17 [00:00<?, ?ba/s]

92518038

In [45]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback
from trl import DPOTrainer, DPOConfig
from peft import get_peft_model, LoraConfig, TaskType
import pandas as pd
import torch
import matplotlib.pyplot as plt

# === Load dữ liệu đã tokenize ===
df_tok = pd.read_json("/kaggle/working/tokenized_dpo.json", lines=True)
tokenized_dataset = Dataset.from_pandas(df_tok)
dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# === Load tokenizer & base model ===
model_name = "EleutherAI/pythia-70m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === Load model gốc cho ref_model (KHÔNG áp dụng LoRA) ===
ref_model = AutoModelForCausalLM.from_pretrained(model_name)

# === Load lại model chính & áp dụng LoRA ===
base_model = AutoModelForCausalLM.from_pretrained(model_name)
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    task_type=TaskType.CAUSAL_LM,
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(base_model, lora_config)

# === Callback ghi logs ===
class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.logs = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            logs['step'] = state.global_step
            self.logs.append(logs)

loss_logger = LossLoggerCallback()

# === Cấu hình huấn luyện ===
training_args = DPOConfig(
    output_dir="/kaggle/working/dpo-output",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy='steps',
    learning_rate = 1e-5
    eval_steps=100,
    save_strategy="no",
    report_to="none",
    beta=0.3,
    max_prompt_length=256,
    max_length=200,
    padding_value=tokenizer.pad_token_id,
)

# === Khởi tạo trainer ===
trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[loss_logger],
)

# === Huấn luyện ===
trainer.train()

# === Lưu model fine-tuned ===
save_path = "/kaggle/working/dpo-finetuned-model"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Extracting prompt in train dataset:   0%|          | 0/15897 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/15897 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15897 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/837 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/837 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/837 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected,
100,2.589600,2.352138,0.191906,0.193218,0.499048,-0.001312,-406.562775,-331.289795,1053.060059,1053.729004,100
200,2.454900,2.391691,-0.088560,-0.046965,0.515952,-0.041595,-407.497620,-332.090454,1053.028687,1053.683350,200
300,2.440500,2.189238,0.281693,0.078865,0.510476,0.202828,-406.263458,-331.671021,1052.986572,1053.665039,300
400,2.534300,2.460202,0.291968,0.118968,0.512857,0.173000,-406.229218,-331.537354,1052.963745,1053.634033,400
500,2.222700,2.429163,0.181975,0.125616,0.492619,0.056359,-406.595825,-331.515167,1052.927368,1053.613159,500
600,2.546900,2.600682,0.001099,0.301062,0.469524,-0.299963,-407.198761,-330.930328,1052.889526,1053.556641,600
700,2.299000,2.452264,0.109634,0.009826,0.506905,0.099808,-406.836975,-331.901154,1052.857422,1053.533691,700
800,2.347100,2.618868,0.221973,0.332781,0.484762,-0.110807,-406.462494,-330.824585,1052.810425,1053.504272,800
900,2.322500,2.490160,0.222199,0.251522,0.501667,-0.029323,-406.461761,-331.095459,1052.781128,1053.476318,900
1000,2.682400,2.361348,0.487690,0.278457,0.523571,0.209232,-405.576782,-331.005737,1052.760010,1053.447266,1000


('/kaggle/working/dpo-finetuned-model/tokenizer_config.json',
 '/kaggle/working/dpo-finetuned-model/special_tokens_map.json',
 '/kaggle/working/dpo-finetuned-model/tokenizer.json')

In [47]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# === Load tokenizer và model gốc (ref_model) ===
model_name = "EleutherAI/pythia-70m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ref_model = AutoModelForCausalLM.from_pretrained(model_name)
ref_model.eval()

# === Load lại model đã fine-tuned bằng DPO ===
dpo_model_path = "/kaggle/working/dpo-finetuned-model"
dpo_model = AutoModelForCausalLM.from_pretrained(dpo_model_path)
dpo_model.eval()

# === Function sinh output ===
def generate_output(model, prompt, max_new_tokens=100, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# === Prompt để test ===
prompt = "What are some good habits to stay productive during the day?"

# === Sinh output từ model gốc và model DPO ===
output_ref = generate_output(ref_model, prompt)
output_dpo = generate_output(dpo_model, prompt)

# === In kết quả so sánh ===
print("🔵 Original (ref_model) output:\n")
print(output_ref)
print("\n🟢 DPO-finetuned model output:\n")
print(output_dpo)


🔵 Original (ref_model) output:

What are some good habits to stay productive during the day?

The next two weeks are at an interesting time in the lab. On the first day, I’m staying in a hotel room or a private room to stay until I’m making an appointment. The rooms are the only one on the premises. There are two main rooms on each side of the room, and I’ve left them to the same staff, which is a bit of a challenge. When I was on the first floor, I was in a good mood. My first

🟢 DPO-finetuned model output:

What are some good habits to stay productive during the day? The answer is to keep it from being the same. You can still eat a bit of meat and drink it while you are in the shower or in the shower.

What are the main reasons why you should stay productive? If you are a new member of the community, you should follow the following:

1. The main reason why you should stay productive during the day? It is the same as the main reason why you should stay productive during the day?

2. T